# Hackathon 5: Model Showdown
## Predicting mortgage denial risk with HMDA data

**Step 1: Frame the problem**

Here we define what we predict and for whom, before touching any data.

- **Target:** whether a home purchase mortgage application is approved or denied
- **Positive class:** denied
- **User:** a housing counselling organisation that decides which applicants get extra help before they apply
- **Affected people:** the applicants the model makes predictions about
- **Worse mistake:** a false negative. Someone who will be denied is told they are fine, gets no help, and loses time, money and maybe the house. A false positive only costs some extra counselling time.
- **Main metric:** recall on the denied class, because it measures how many of the people at risk we actually catch. We watch precision next to it, so the model cannot cheat by flagging everyone.

**Step 2: Load the data**

Here we download one slice of the public HMDA data from the CFPB Data Browser: one state, one year, home purchase loans, and only applications that were approved or denied. The raw file is saved in a `data` folder, so the download only happens once and every run starts from the exact same file.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import pandas as pd

# here we choose which slice of hmda we use
STATE = "DE"   # delaware, a small state, keeps the file manageable
YEAR = 2023

# actions_taken: 1 = loan originated (approved), 3 = application denied
# loan_purposes: 1 = home purchase
URL = (
    "https://ffiec.cfpb.gov/v2/data-browser-api/view/csv"
    f"?states={STATE}&years={YEAR}&actions_taken=1,3&loan_purposes=1"
)

DATA_PATH = Path("data") / f"hmda_{STATE}_{YEAR}_purchase.csv"

# here we only download if the file is not there yet
if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(exist_ok=True)
    urlretrieve(URL, DATA_PATH)

# here we load the raw data into a dataframe
df_raw = pd.read_csv(DATA_PATH, low_memory=False)

print("rows, columns:", df_raw.shape)
print(df_raw["action_taken"].value_counts())
df_raw.head()

**Step 3: Explore the data, part 1: balance, duplicates and leakage**

Here we create our target column (1 = denied, 0 = approved) and check how balanced it is. Then we look for duplicate rows, and we compare how often each column is missing for approved versus denied applications. A column that is almost always empty for one group and filled for the other gives away the answer, which means it is only known after the decision.

In [ ]:
# here we create the target: 1 = denied, 0 = approved
df_raw["denied"] = (df_raw["action_taken"] == 3).astype(int)

# here we check the class balance in percentages
print(df_raw["denied"].value_counts(normalize=True).round(3))

# here we count exact duplicate rows
print("duplicate rows:", df_raw.duplicated().sum())

# here we calculate the % missing per column, separately for approved and denied
missing = (
    df_raw.isna()
    .groupby(df_raw["denied"])
    .mean()
    .T
    .mul(100)
    .round(1)
)
missing.columns = ["% missing approved", "% missing denied"]

# here we sort on the biggest difference between the two groups
missing["difference"] = (missing["% missing denied"] - missing["% missing approved"]).abs()
missing.sort_values("difference", ascending=False).head(25)

**Step 3: Explore the data, part 2: duplicates and leakage through values**

Here we remove the 13 exact duplicate rows, so no copy can end up in both the training and the test set. Then we look at a few code columns that are always filled in, but might still give away the answer through *which* code they contain. For each column we show the share of each code within approved and within denied applications.

In [ ]:
# here we drop exact duplicate rows
df = df_raw.drop_duplicates().copy()
print("rows after removing duplicates:", len(df))

# here we list code columns that are filled in but might leak through their values
suspects = [
    "denial_reason-1",
    "purchaser_type",
    "hoepa_status",
    "aus-1",
    "applicant_credit_score_type",
    "initially_payable_to_institution",
    "submission_of_application",
]

# here we show, per column, the % of each code within approved (0) and denied (1)
for col in suspects:
    print(f"\n--- {col} ---")
    print(pd.crosstab(df[col], df["denied"], normalize="columns").mul(100).round(1))

**Step 3: Explore the data, part 3: select usable columns**

Here we stop removing bad columns one by one and instead keep only the columns a counselling organisation knows before someone applies. Everything else is dropped: it either leaks the outcome (pricing, denial reasons, lender process codes), identifies the lender or location, or repeats information we already have. Sex, race, ethnicity and age are kept aside, not as model input, but to check later whether the model makes more mistakes for some groups. Finally we turn HMDA's hidden missing values ("Exempt" and code 1111) into real missing values.

In [ ]:
import numpy as np

# here we choose the columns known before someone applies
NUM_COLS = ["loan_amount", "property_value", "income", "loan_term",
            "tract_to_msa_income_percentage"]
CAT_COLS = ["loan_type", "lien_status", "occupancy_type", "construction_method",
            "total_units", "preapproval", "submission_of_application",
            "conforming_loan_limit", "debt_to_income_ratio"]

# here we keep sensitive columns aside, only for the fairness check later
GROUP_COLS = ["derived_sex", "derived_race", "derived_ethnicity", "applicant_age"]
TARGET = "denied"

df = df[NUM_COLS + CAT_COLS + GROUP_COLS + [TARGET]].copy()

# here we turn text like "Exempt" in number columns into real missing values
for col in NUM_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# here we turn the exempt codes in category columns into real missing values
df[CAT_COLS] = df[CAT_COLS].replace({1111: np.nan, "Exempt": np.nan})

print("shape:", df.shape)
print("\n% missing per column:")
print(df.isna().mean().mul(100).round(1).sort_values(ascending=False))
df[NUM_COLS].describe().round(1)

**Step 3: Explore the data, part 4: impossible values and skewed distributions**

Here we turn impossible values into missing values: a negative income, and a neighbourhood income percentage of 0, which means no data. This is a fixed rule that does not learn anything from the data, so it is safe to do before the split. Then we plot the three money columns as they are and after a log transform, to see whether a few extreme values dominate.

In [ ]:
import matplotlib.pyplot as plt

# here we count the impossible values before fixing them
print("income <= 0:", (df["income"] <= 0).sum())
print("tract income % = 0:", (df["tract_to_msa_income_percentage"] == 0).sum())

# here we turn the impossible values into missing values
df.loc[df["income"] <= 0, "income"] = np.nan
df.loc[df["tract_to_msa_income_percentage"] == 0, "tract_to_msa_income_percentage"] = np.nan

# here we plot the money columns raw (top) and after a log transform (bottom)
money_cols = ["loan_amount", "property_value", "income"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for i, col in enumerate(money_cols):
    values = df[col].dropna()
    axes[0, i].hist(values, bins=50)
    axes[0, i].set_title(f"{col} (raw)")
    axes[1, i].hist(np.log1p(values), bins=50)
    axes[1, i].set_title(f"{col} (log)")

plt.tight_layout()
plt.show()

**Step 3: Explore the data, part 5: who is in the data**

Here we look at the sensitive columns we kept aside: how big each group is in the data, and how often each group was denied. This tells us who is well represented and who is not, and whether denial rates already differ between groups before any model is involved. We use this later for the fairness check and the ethical reflection.

In [ ]:
# here we show, per group, its size and its denial rate
for col in GROUP_COLS:
    summary = df.groupby(col)["denied"].agg(["count", "mean"])
    summary["share of data %"] = (summary["count"] / len(df) * 100).round(1)
    summary["denial rate %"] = (summary["mean"] * 100).round(1)

    print(f"\n--- {col} ---")
    print(summary[["count", "share of data %", "denial rate %"]]
          .sort_values("count", ascending=False))

**Step 3: Summary of the exploration**

- 15,762 applications after removing 13 duplicates, 15.4% denied. The data is unbalanced, so accuracy is not a useful metric: always predicting "approved" already scores about 85%.
- Many columns leak the outcome, either through missing values (pricing only exists for approved loans) or through "not applicable" codes (denial reasons, purchaser type). We kept only 14 features known before someone applies.
- Missing values are small (5% or less) and are filled in inside the Pipeline. Impossible values (negative income, a neighbourhood income of 0) were turned into missing values.
- The money columns are extremely skewed, so they get a log transform inside the Pipeline.
- Historical denial rates differ strongly per group: Black applicants were denied twice as often as White applicants (23.6% vs 11.6%), Hispanic applicants 22.7% vs 13.5%. The model learns from these past decisions, so we check its mistakes per group later. Several groups are too small (under 100 people) to evaluate reliably.

**Step 4: Split first**

Here we set aside 20% of the data as the test set, before any filling in, scaling or tuning. We stratify on the target so both sets have the same share of denied applications, and fix the random_state so every run gives the same split. The test set is used exactly once, at the very end. The group columns are split along with the rest, so we can check fairness on the test set later, but they are not part of the model input.

In [ ]:
from sklearn.model_selection import train_test_split

# here we split the whole dataframe, so group columns stay aligned with each row
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df[TARGET],
    random_state=42,
)

# here we separate model input (X) from the answer (y)
FEATURES = NUM_COLS + CAT_COLS
X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

print("train:", X_train.shape, " test:", X_test.shape)
print("denied % train:", round(y_train.mean() * 100, 1))
print("denied % test: ", round(y_test.mean() * 100, 1))

**Step 5: Pre-process inside a Pipeline**

Here we build all pre-processing as one scikit-learn Pipeline, so every step learns only from the training data. Money columns get their missing values filled with the median, a log transform and scaling. The other number columns get the median and scaling. Category columns get the most common value for missing cells and are one-hot encoded. Scaling is needed because KNN measures distances and logistic regression compares the weights of features, and both only work fairly when all features are on the same scale. We fit it once here only to check the output; later it is fitted again inside every cross-validation fold.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

MONEY_COLS = ["loan_amount", "property_value", "income"]
OTHER_NUM_COLS = ["loan_term", "tract_to_msa_income_percentage"]

# here we define the lane for money columns: fill, log, scale
money_steps = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scale", StandardScaler()),
])

# here we define the lane for the other number columns: fill, scale
num_steps = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

# here we define the lane for category columns: fill, one-hot
cat_steps = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# here we send each group of columns to its own lane
preprocessor = ColumnTransformer([
    ("money", money_steps, MONEY_COLS),
    ("num", num_steps, OTHER_NUM_COLS),
    ("cat", cat_steps, CAT_COLS),
])

# here we only check that it works, the real fitting happens later per fold
X_check = preprocessor.fit_transform(X_train)
print("shape after pre-processing:", X_check.shape)
print(preprocessor.get_feature_names_out()[:12])

**Step 6: Set a baseline**

Here we fix the cross-validation folds that every model will use: 5 stratified folds on the training set, with a fixed random_state. Then we score a DummyClassifier that always predicts the most common class (approved). Every real model has to beat this on recall. We also show its accuracy, to make visible why accuracy is misleading on this data.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.metrics import make_scorer, precision_score

# here we fix the 5 folds, every model uses exactly these folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# here we define the metrics, precision gives 0 instead of a warning when nothing is flagged
SCORING = {
    "recall": "recall",
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": "f1",
    "accuracy": "accuracy",
}

# here we put the baseline behind the same pre-processing as the real models
baseline = Pipeline([
    ("prep", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent")),
])

# here we score the baseline on the 5 folds
scores = cross_validate(baseline, X_train, y_train, cv=cv, scoring=SCORING)

for metric in SCORING:
    s = scores[f"test_{metric}"]
    print(f"{metric:10s} {s.mean():.3f} ± {s.std():.3f}")

**Step 6: Set a baseline, part 2: a simple rule**

The DummyClassifier is easy to beat on recall, because it never flags anyone. A more honest comparison is a rule a counsellor could use without any machine learning: flag every applicant with a debt-to-income ratio of 50% or higher. This rule learns nothing from the data, so we can score it directly on the whole training set.

In [ ]:
from sklearn.metrics import recall_score, f1_score

# here we apply the simple rule: high debt-to-income means flagged as at risk
HIGH_DTI = ["50%-60%", ">60%"]
rule_pred = X_train["debt_to_income_ratio"].isin(HIGH_DTI).astype(int)

print("flagged %: ", round(rule_pred.mean() * 100, 1))
print("recall:    ", round(recall_score(y_train, rule_pred), 3))
print("precision: ", round(precision_score(y_train, rule_pred, zero_division=0), 3))
print("f1:        ", round(f1_score(y_train, rule_pred), 3))

**Step 6: Set a baseline, part 3: how we pick the best setting**

Choosing the setting with the highest recall alone rewards models that simply flag more people, because flagging everyone gives a recall of 1.0. So we pick the best setting with one extra condition: the highest recall among the settings whose precision is at least as good as the simple rule. A counselling organisation then catches as many people at risk as possible, without more false alarms than the simple rule would give. All three models use this same rule.

In [ ]:
# here we store the precision of the simple rule as the minimum a model must reach
RULE_PRECISION = precision_score(y_train, rule_pred)

# here we pick the highest recall among settings with at least the rule's precision
def pick_best(cv_results):
    res = pd.DataFrame(cv_results)
    allowed = res[res["mean_test_precision"] >= RULE_PRECISION]
    if allowed.empty:
        # here we fall back to plain recall if no setting reaches the floor
        return int(res["mean_test_recall"].idxmax())
    return int(allowed["mean_test_recall"].idxmax())

print("precision floor:", round(RULE_PRECISION, 3))

**Step 7: Tune with cross-validation, model 1: KNN**

Here we tune a k-nearest-neighbours model on the training set, using the same 5 folds and metrics as the baseline. We try different numbers of neighbours and two ways of weighting their votes, and pick the combination with the highest mean recall. We plot recall against the number of neighbours, with the training score next to it to see over- and underfitting. A small helper function collects the results of each model for the comparison table later.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

# here we collect the cross-validation results of the best setting of a search
def cv_summary(search):
    res = pd.DataFrame(search.cv_results_)
    i = search.best_index_
    summary = {"best params": search.best_params_}
    for metric in ["recall", "precision", "f1"]:
        summary[f"cv {metric}"] = (
            f"{res.loc[i, f'mean_test_{metric}']:.3f} ± {res.loc[i, f'std_test_{metric}']:.3f}"
        )
    summary["train recall"] = round(res.loc[i, "mean_train_recall"], 3)
    return summary

# here we combine pre-processing and the model in one pipeline
knn_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", KNeighborsClassifier()),
])

# here we list every setting we want to try
knn_grid = {
    "model__n_neighbors": [1, 3, 5, 7, 11, 15, 21, 31],
    "model__weights": ["uniform", "distance"],
}

# here we try every combination on the same 5 folds and pick the best on recall
knn_search = GridSearchCV(
    knn_pipe, knn_grid, cv=cv, scoring=SCORING,
    refit=pick_best, return_train_score=True, n_jobs=-1,
)
knn_search.fit(X_train, y_train)

print(cv_summary(knn_search))

# here we plot recall against k: validation for both weightings, train for uniform
res = pd.DataFrame(knn_search.cv_results_)
fig, ax = plt.subplots(figsize=(9, 4))

for w in ["uniform", "distance"]:
    r = res[res["param_model__weights"] == w]
    ax.errorbar(r["param_model__n_neighbors"], r["mean_test_recall"],
                yerr=r["std_test_recall"], marker="o", capsize=3,
                label=f"{w} (validation)")

r = res[res["param_model__weights"] == "uniform"]
ax.plot(r["param_model__n_neighbors"], r["mean_train_recall"],
        linestyle="--", label="uniform (train)")

ax.axhline(0.466, color="grey", linestyle=":", label="simple rule")
ax.set_xlabel("n_neighbors (k)")
ax.set_ylabel("recall (denied)")
ax.set_title("KNN: recall against number of neighbours")
ax.legend()
plt.show()

**Step 7: Check for twin applications**

KNN works best with only one neighbour, which is suspicious. One person often applies to several lenders for the same house, which creates rows that are identical on all our features. If such twins end up on both sides of a split, the model can simply look up the answer. Here we count how many rows have an identical twin on all 14 features, within the training set and between test and training set, and whether twins usually share the same outcome.

In [ ]:
# here we turn every row into one text key, so identical rows get identical keys
train_key = X_train.astype(str).agg("|".join, axis=1)
test_key = X_test.astype(str).agg("|".join, axis=1)

print("train rows with a twin in train:", train_key.duplicated(keep=False).sum())
print("test rows with a twin in train: ", test_key.isin(train_key).sum())

# here we check how often twins share the same outcome
twins = pd.DataFrame({"key": train_key, "denied": y_train})
twins = twins[twins["key"].duplicated(keep=False)]
same_outcome = twins.groupby("key")["denied"].nunique().eq(1).mean()

print("twin groups with the same outcome %:", round(same_outcome * 100, 1))
print("denial rate among twins %:          ", round(twins["denied"].mean() * 100, 1))

**Step 7: Tune with cross-validation, model 2: logistic regression**

The twin check found only 46 training rows and 11 test rows with an identical twin, and twins share the same outcome only about half the time, so this does not explain KNN's results. We keep the data as it is. KNN works best with one neighbour because every extra neighbour adds mostly "approved" votes.

Here we tune a logistic regression on the same folds and metrics. We try a range of values for C, which controls how much freedom the model gets, with and without class balancing, which makes mistakes on the rare denied class count more. We plot recall and precision against C for both settings.

In [ ]:
from sklearn.linear_model import LogisticRegression

# here we start the collection of results with knn
results = {"KNN": cv_summary(knn_search)}

lr_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])

# here we list the settings to try, C on a log scale
lr_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"],
}

lr_search = GridSearchCV(
    lr_pipe, lr_grid, cv=cv, scoring=SCORING,
    refit=pick_best, return_train_score=True, n_jobs=-1,
)
lr_search.fit(X_train, y_train)

results["Logistic regression"] = cv_summary(lr_search)
print(results["Logistic regression"])

# here we plot recall and precision against C, one panel per class_weight
res = pd.DataFrame(lr_search.cv_results_)
res["cw"] = res["param_model__class_weight"].astype(str)
res["C"] = res["param_model__C"].astype(float)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, cw in zip(axes, ["None", "balanced"]):
    r = res[res["cw"] == cw]
    for metric in ["recall", "precision"]:
        ax.errorbar(r["C"], r[f"mean_test_{metric}"], yerr=r[f"std_test_{metric}"],
                    marker="o", capsize=3, label=f"{metric} (validation)")
    ax.plot(r["C"], r["mean_train_recall"], linestyle="--", label="recall (train)")
    ax.axhline(0.466, color="grey", linestyle=":", label="simple rule recall")
    ax.set_xscale("log")
    ax.set_xlabel("C")
    ax.set_title(f"class_weight = {cw}")

axes[0].set_ylabel("score")
axes[0].legend()
plt.show()

**Step 7: Tune with cross-validation, model 3: decision tree**

Our own choice is a decision tree, because its decisions can be read as a set of if-then rules, which matters for a counselling organisation that has to explain a flag to an applicant. Here we tune the maximum depth, the minimum number of applicants per leaf and class balancing, on the same folds and metrics. A tree does not need scaling, because every split looks at one column at a time, but we keep the same pre-processing so all three models get exactly the same input. We plot recall and precision against the depth.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42)),
])

# here we list the settings to try
tree_grid = {
    "model__max_depth": [2, 3, 4, 5, 6, 8, 10, 15, None],
    "model__min_samples_leaf": [1, 20, 50],
    "model__class_weight": [None, "balanced"],
}

tree_search = GridSearchCV(
    tree_pipe, tree_grid, cv=cv, scoring=SCORING,
    refit=pick_best, return_train_score=True, n_jobs=-1,
)
tree_search.fit(X_train, y_train)

results["Decision tree"] = cv_summary(tree_search)
print(results["Decision tree"])

# here we plot recall and precision against depth, at the best min_samples_leaf
res = pd.DataFrame(tree_search.cv_results_)
res["cw"] = res["param_model__class_weight"].astype(str)
res["depth"] = res["param_model__max_depth"].astype(str)
best_leaf = tree_search.best_params_["model__min_samples_leaf"]
depth_order = ["2", "3", "4", "5", "6", "8", "10", "15", "None"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, cw in zip(axes, ["None", "balanced"]):
    r = res[(res["cw"] == cw) & (res["param_model__min_samples_leaf"] == best_leaf)]
    r = r.set_index("depth").loc[depth_order]
    for metric in ["recall", "precision"]:
        ax.errorbar(depth_order, r[f"mean_test_{metric}"], yerr=r[f"std_test_{metric}"],
                    marker="o", capsize=3, label=f"{metric} (validation)")
    ax.plot(depth_order, r["mean_train_recall"], linestyle="--", label="recall (train)")
    ax.axhline(0.466, color="grey", linestyle=":", label="simple rule recall")
    ax.set_xlabel("max_depth")
    ax.set_title(f"class_weight = {cw}, min_samples_leaf = {best_leaf}")

axes[0].set_ylabel("score")
axes[0].legend()
plt.show()

**Step 8: Evaluate once on the test set, part 1: the comparison table**

Here we use the test set for the first and only time. Every tuned model, the DummyClassifier and the simple rule predict the 3,153 test applications they have never seen. The table puts the training score, the cross-validation score and the test score next to each other per model. Nothing in the models is changed after this step.

In [ ]:
# here we score predictions on the test set
def test_scores(y_true, y_pred):
    return {
        "test recall": round(recall_score(y_true, y_pred), 3),
        "test precision": round(precision_score(y_true, y_pred, zero_division=0), 3),
        "test f1": round(f1_score(y_true, y_pred), 3),
    }

# here we run the baseline through an empty search, so it gets the same summary as the models
base_search = GridSearchCV(baseline, {}, cv=cv, scoring=SCORING,
                           refit=pick_best, return_train_score=True)
base_search.fit(X_train, y_train)

searches = {
    "Baseline (most frequent)": base_search,
    "KNN": knn_search,
    "Logistic regression": lr_search,
    "Decision tree": tree_search,
}

# here we predict the test set once per model and collect all scores
test_preds = {}
rows = []
for name, search in searches.items():
    pred = search.best_estimator_.predict(X_test)
    test_preds[name] = pred
    rows.append({"model": name, **cv_summary(search), **test_scores(y_test, pred)})

# here we add the simple rule, which has no cross-validation because it learns nothing
rule_test = X_test["debt_to_income_ratio"].isin(HIGH_DTI).astype(int)
test_preds["Simple rule (DTI >= 50%)"] = rule_test
rows.insert(1, {
    "model": "Simple rule (DTI >= 50%)",
    "best params": "-",
    "train recall": round(recall_score(y_train, rule_pred), 3),
    "cv recall": "-", "cv precision": "-", "cv f1": "-",
    **test_scores(y_test, rule_test),
})

columns = ["best params", "train recall", "cv recall", "cv precision", "cv f1",
           "test recall", "test precision", "test f1"]
comparison = pd.DataFrame(rows).set_index("model")[columns]
comparison

**Step 8: Evaluate once on the test set, part 2: confusion matrices**

Here we show a confusion matrix per model on the test set. The bottom-left cell is the most important one for our user: applicants who were denied but whom the model did not flag, so they would not have received help.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# here we draw one confusion matrix per model on the test set
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, pred) in zip(axes, test_preds.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=["approved", "denied"],
        colorbar=False, ax=ax,
    )
    ax.set_title(name, fontsize=10)

plt.tight_layout()
plt.show()

**Step 8: Result**

Test scores are within 0.01 of the cross-validation scores for every model, so the cross-validation estimates were reliable and nothing leaked into tuning. KNN overfits strongly (train recall 0.998, test 0.555). Logistic regression shows no overfitting (0.703 train, 0.696 CV, 0.699 test), and the decision tree only a small gap. All models top out around 0.70 recall, which points to information that is missing from the data, most likely the credit score. On the 485 denied applicants in the test set, logistic regression and the decision tree each catch about 110 more people than the simple rule, at the cost of 100 to 130 extra false alarms. Logistic regression and the decision tree are practically tied.

**Step 9: Look at the errors, part 1: who is missed**

Here we compare the denied applicants that logistic regression and the decision tree caught with the ones they missed. We look at their median income and loan amount, how many had a high debt-to-income ratio, and how many had no debt-to-income ratio at all. This shows what kind of person falls through the cracks.

In [ ]:
# here we compare caught and missed denied applicants for the two best models
denied_rows = test_df[y_test == 1]
rows = []

for name in ["Logistic regression", "Decision tree"]:
    pred = pd.Series(test_preds[name], index=X_test.index)[y_test == 1]
    for label, mask in [("caught", pred == 1), ("missed", pred == 0)]:
        part = denied_rows[mask]
        rows.append({
            "model": name,
            "denied applicants": label,
            "count": len(part),
            "median income (K)": part["income"].median(),
            "median loan": part["loan_amount"].median(),
            "high DTI %": round(part["debt_to_income_ratio"].isin(HIGH_DTI).mean() * 100, 1),
            "DTI missing %": round(part["debt_to_income_ratio"].isna().mean() * 100, 1),
        })

pd.DataFrame(rows).set_index(["model", "denied applicants"])

**Step 9: Look at the errors, part 2: performance per group**

Here we check whether the two best models make the same kind of mistakes for different groups, using the sensitive columns we kept aside. Per group we show the recall (the share of denied applicants that were caught) and the false alarm rate (the share of approved applicants that were flagged anyway). We only compare groups that are large enough in the test set: White and Black applicants, Hispanic and non-Hispanic applicants, and male, female and joint applications.

In [ ]:
# here we calculate recall and false alarm rate for each group
def group_report(pred, col, groups):
    pred = pd.Series(pred, index=X_test.index)
    rows = []
    for g in groups:
        m = test_df[col] == g
        y_g, p_g = y_test[m], pred[m]
        false_alarms = ((p_g == 1) & (y_g == 0)).sum()
        rows.append({
            "group": g,
            "applicants": int(m.sum()),
            "denied": int(y_g.sum()),
            "recall": round(recall_score(y_g, p_g), 3),
            "false alarm rate": round(false_alarms / (y_g == 0).sum(), 3),
        })
    return rows

checks = {
    "derived_race": ["White", "Black or African American"],
    "derived_ethnicity": ["Not Hispanic or Latino", "Hispanic or Latino"],
    "derived_sex": ["Male", "Female", "Joint"],
}

# here we run the group check for both models
all_rows = []
for name in ["Logistic regression", "Decision tree"]:
    for col, groups in checks.items():
        for r in group_report(test_preds[name], col, groups):
            all_rows.append({"model": name, "column": col, **r})

pd.DataFrame(all_rows).set_index(["model", "column", "group"])

**Step 9: Result**

The denied applicants that both models miss look financially healthy on paper: a median income of about 100K instead of 60K, larger loans, and rarely a high debt-to-income ratio. They were most likely denied for reasons that are not in the data, such as credit history or the property itself. Missed applicants also have a missing debt-to-income ratio four times as often (15% vs 4%). Because we fill a missing value with the most common category, these applicants look average to the model. A missing-value indicator could help, but we do not change the model after using the test set; this is noted as a next step.

Recall is similar for all groups (0.68 to 0.75), so no group is missed more often. The false alarm rate is not: logistic regression flags approved Black and Hispanic applicants about 1.5 times as often as White and non-Hispanic applicants (0.194 vs 0.127, 0.196 vs 0.131). The decision tree reduces this gap to about 1.2 times (0.150 vs 0.127, 0.143 vs 0.122). Race is not a model input, so this difference comes in through related columns such as income and neighbourhood income. Groups are small in the test set (55 denied Hispanic applicants), so these numbers are indicative, not exact.

**Step 10: Recommend, part 1: what drives the decision tree**

Here we look at which features the recommended model relies on most. A feature's importance shows how much it helped to separate denied from approved applications across all splits in the tree.

In [ ]:
# here we take the tuned tree and the feature names after pre-processing
tree_model = tree_search.best_estimator_
names = tree_model.named_steps["prep"].get_feature_names_out()
importances = pd.Series(tree_model.named_steps["model"].feature_importances_, index=names)

# here we plot the 10 most important features
importances.sort_values().tail(10).plot.barh(figsize=(8, 4))
plt.title("Decision tree: 10 most important features")
plt.xlabel("importance")
plt.show()

**Step 10: Recommend, part 2: our recommendation**

We recommend the **decision tree** (max_depth 10, min_samples_leaf 50, balanced class weights) to the housing counselling organisation.

- **Recall:** it catches 69% of the applicants who will be denied (336 of 485 in the test set), against 47% for the simple debt-to-income rule. That is about 110 more people who get help in time.
- **Not a clear winner on numbers:** logistic regression scores the same recall (0.696 vs 0.688 in cross-validation), and the difference is far smaller than the spread over the folds (± 0.05). So the choice is made on the other criteria.
- **Fewer false alarms:** 347 against 379 for logistic regression.
- **Fairer mistakes:** the gap in false alarms between Black and White applicants is 0.150 vs 0.127, against 0.194 vs 0.127 for logistic regression.
- **Explainable:** its decisions are if-then rules, so a counsellor can explain why someone was flagged.
- **Not KNN:** it overfits (train recall 0.998 vs test 0.555) and catches fewer people.

**Conditions of use:** the model may only be used to *offer* extra help, never to refuse anything to anyone. It is trained on home purchase applications in Delaware in 2023 and must not be used for other states, years or loan types without retraining. One in three people at risk is still missed, mostly applicants with good income on paper, so counsellors should not tell anyone "you are safe" based on this model.

**Step 11: Use it**

Here we enter two made-up applicants and let the recommended model predict whether they are at risk of being denied. We copy an existing test row only as a template, so every column has the right format, and then fill in our own values. The risk score is the share of denied applications in the tree leaf the applicant ends up in. Because we used balanced class weights, this score is pushed upwards: it ranks applicants by risk, but it is not a real probability.

In [ ]:
model = tree_search.best_estimator_

# here we copy one test row twice as a template, so all columns have the right format
new_cases = pd.concat([X_test.iloc[[0]]] * 2, ignore_index=True)
cols = ["income", "loan_amount", "property_value", "debt_to_income_ratio", "loan_term"]

# here we fill in two made-up applicants
new_cases.loc[0, cols] = [45, 180000, 200000, "50%-60%", 360]    # low income, high debt
new_cases.loc[1, cols] = [140, 300000, 450000, "20%-<30%", 360]  # high income, low debt

# here we let the model predict
pred = model.predict(new_cases)
risk = model.predict_proba(new_cases)[:, 1]

for i, label in enumerate(["Applicant A", "Applicant B"]):
    status = "AT RISK, offer counselling" if pred[i] == 1 else "not flagged"
    print(f"{label}: {status} (risk score {risk[i]:.2f})")